In [1]:
# TinyLlama braucht keine spezielle transformers Version
# LlamaForCausalLM ist eine der ältesten Architekturen -> keine Kompatibilitätsprobleme
!pip install transformers accelerate bitsandbytes sentence-transformers faiss-cpu -q
print("Fertig!")

Fertig!


In [2]:
from llama_cpp import Llama

# __version__ und CUDA Support prüfen
import llama_cpp
print(f"llama-cpp-python Version: {llama_cpp.__version__}")

# Kleiner Test ob CUDA erkannt wird
import ctypes
try:
    ctypes.CDLL("libcuda.so")
    print("CUDA verfügbar")
except:
    print("CUDA nicht verfügbar - läuft auf CPU")

llama-cpp-python Version: 0.3.20
CUDA verfügbar


In [3]:
import json
import pandas as pd
import numpy as np
import torch

# Chunks laden - unsere 6685 Gesetzesabschnitte die wir lokal vorbereitet haben
with open("/kaggle/input/datasets/anso1234/alle-chunks/alle_chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)
print(f"Chunks geladen: {len(chunks)}")

# Dataset mit den 644 Fragen direkt von GitHub laden
dataset_url = "https://raw.githubusercontent.com/svakulenk0/wu-llms-ss26/main/dataset_clean.csv"
df = pd.read_csv(dataset_url)
print(f"Dataset geladen: {len(df)} Fragen")
print(f"Spalten: {df.columns.tolist()}")
print(f"\nErste Zeile:")
print(df.iloc[0])

Chunks geladen: 6685
Dataset geladen: 643 Fragen
Spalten: ['id', 'prompt']

Erste Zeile:
id                                             CORP-TAX-001
prompt    Was ist die steuerliche Bemessungsgrundlage fü...
Name: 0, dtype: object


In [4]:
from sentence_transformers import SentenceTransformer
import faiss

# Sentence Transformer laden
# multilingual-e5-base versteht Deutsch und misst semantische Ähnlichkeit
# d.h. er findet Chunks die inhaltlich zur Frage passen, nicht nur Stichwörter
print("Lade Retriever...")
retriever = SentenceTransformer("intfloat/multilingual-e5-base")
print("Retriever geladen")

# Alle 6685 Chunk-Texte in Vektoren umwandeln
# Jeder Chunk wird als Liste von Zahlen dargestellt
# Ähnliche Texte haben ähnliche Zahlen -> FAISS kann schnell suchen
print("\nWandle Chunks in Vektoren um...")
chunk_texte = [c['text'] for c in chunks]

chunk_vektoren = retriever.encode(
    chunk_texte,
    show_progress_bar=True,
    batch_size=64,          # 64 Chunks gleichzeitig verarbeiten
    convert_to_numpy=True   # FAISS braucht numpy arrays
)

print(f"Vektoren erstellt: {chunk_vektoren.shape}")

# FAISS Index aufbauen
# Vektoren normalisieren damit die Suche korrekt funktioniert
dimension = chunk_vektoren.shape[1]
faiss.normalize_L2(chunk_vektoren)

# Index erstellen und alle Vektoren einfügen
index = faiss.IndexFlatIP(dimension)
index.add(chunk_vektoren)

print(f"FAISS Index aufgebaut mit {index.ntotal} Vektoren")

Lade Retriever...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retriever geladen

Wandle Chunks in Vektoren um...


Batches:   0%|          | 0/105 [00:00<?, ?it/s]

Vektoren erstellt: (6685, 768)
FAISS Index aufgebaut mit 6685 Vektoren


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# TinyLlama-1.1B-Chat:
# - 1.1 Milliarden Parameter, sehr klein und schnell
# - LlamaForCausalLM Architektur: funktioniert mit jeder Transformers Version
# - Chat-Version: versteht Anweisungen auf Deutsch
# - passt problemlos auf T4 GPU

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit Quantisierung: Modell von ~2GB auf ~0.7GB reduzieren
# so bleibt viel GPU Speicher für den Gesetzestext-Kontext
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Lade Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Lade Modell auf GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="cuda:0"   # explizit auf GPU 0 laden
)

print("Modell geladen")
print(f"GPU Speicher verwendet: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")

# Kurzer Test ob alles funktioniert
test_input = tokenizer("Hallo!", return_tensors="pt").to("cuda:0")
with torch.no_grad():
    test_output = model.generate(**test_input, max_new_tokens=10)
print(f"Test: {tokenizer.decode(test_output[0], skip_special_tokens=True)}")

Lade Tokenizer...
Lade Modell auf GPU...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modell geladen
GPU Speicher verwendet: 1.9 GB
Test: Hallo!

(Sings)

(S


In [6]:
import re
import time

# -------------------------------------------------------
# Funktion 1: Relevante Chunks finden
# -------------------------------------------------------

def retrieve_chunks(frage, top_k=5):
    """
    Für eine Frage die top_k relevantesten Gesetzesabschnitte finden.
    Ablauf: Frage -> Vektor -> FAISS Suche -> Chunks zurückgeben
    """
    
    # Frage in Vektor umwandeln
    # "query: " Prefix ist für multilingual-e5 wichtig
    frage_vektor = retriever.encode(
        ["query: " + frage],
        convert_to_numpy=True
    )
    faiss.normalize_L2(frage_vektor)
    
    # FAISS sucht die ähnlichsten Vektoren
    scores, indices = index.search(frage_vektor, top_k)
    
    # Chunks anhand der gefundenen Indices zurückgeben
    return [chunks[idx] for idx in indices[0] if idx != -1]


# -------------------------------------------------------
# Funktion 2: Antwort mit TinyLlama generieren
# -------------------------------------------------------

def generate_answer(frage, relevante_chunks):
    """
    Kontext aus Chunks + Frage als Prompt an TinyLlama schicken.
    TinyLlama antwortet basierend auf den Gesetzestexten - das ist RAG.
    """
    
    # Alle relevanten Chunks zu einem Kontext zusammenfügen
    kontext_teile = []
    for chunk in relevante_chunks:
        # Quelle mitangeben damit das Modell weiß woher der Text stammt
        kontext_teile.append(f"[{chunk['quelle']}]\n{chunk['text']}")
    kontext = "\n\n".join(kontext_teile)
    
    # TinyLlama Chat Format mit <|system|> Tags
    prompt = (
        "<|system|>\n"
        "Du bist ein Experte für österreichisches Steuerrecht. "
        "Beantworte die Frage kurz und präzise auf Deutsch, "
        "ausschließlich basierend auf den Gesetzestexten.</s>\n"
        "<|user|>\n"
        f"Kontext:\n{kontext}\n\n"
        f"Frage: {frage}</s>\n"
        "<|assistant|>\n"
    )
    
    # Tokenisieren und auf GPU schieben
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2000    # Kontextlimit nicht überschreiten
    ).to("cuda:0")
    
    # Antwort generieren
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,              # kurze Antwort reicht
            temperature=0.1,                 # niedrig = präzise Antworten
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Nur die neuen Tokens dekodieren, nicht den gesamten Prompt
    neue_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    antwort = tokenizer.decode(neue_tokens, skip_special_tokens=True)
    
    return antwort.strip()


# -------------------------------------------------------
# Funktion 3: Antwort für CSV bereinigen
# -------------------------------------------------------

def clean_answer(antwort):
    """
    Zeilenumbrüche entfernen damit die CSV-Datei sauber bleibt.
    Eine Antwort muss in einer einzigen Zeile stehen.
    """
    antwort = antwort.replace("\n", " ")
    antwort = re.sub(r'\s+', ' ', antwort)
    return antwort.strip()


# -------------------------------------------------------
# Prompt-Spalte automatisch finden
# -------------------------------------------------------

prompt_spalte = None
for col in df.columns:
    if col.lower() in ['prompt', 'question', 'frage', 'text']:
        prompt_spalte = col
        break
if prompt_spalte is None:
    prompt_spalte = df.columns[1]  # zweite Spalte als Fallback

print(f"Prompt-Spalte: '{prompt_spalte}'")
print(f"Beispiel: {df[prompt_spalte].iloc[0][:100]}\n")


# -------------------------------------------------------
# Hauptschleife: alle 644 Fragen beantworten
# -------------------------------------------------------

print(f"Starte RAG Pipeline für {len(df)} Fragen...")

ergebnisse = []
fehler_count = 0
start_time = time.time()

for i, row in df.iterrows():
    
    frage_id   = row['id']
    frage_text = row[prompt_spalte]
    
    try:
        relevante_chunks = retrieve_chunks(frage_text, top_k=5)
        antwort = generate_answer(frage_text, relevante_chunks)
        antwort_sauber = clean_answer(antwort)
        
    except Exception as e:
        fehler_count += 1
        print(f"  Fehler bei {frage_id}: {e}")
        antwort_sauber = "Fehler bei der Verarbeitung"
    
    ergebnisse.append({"id": frage_id, "answer": antwort_sauber})

    # Nach jeder 5. Frage Status ausgeben
    if (i + 1) % 5 == 0:
        elapsed      = time.time() - start_time
        done         = i + 1
        remaining    = len(df) - done
        per_question = elapsed / done
        eta_seconds  = remaining * per_question

        elapsed_str  = time.strftime("%H:%M:%S", time.gmtime(elapsed))
        eta_str      = time.strftime("%H:%M:%S", time.gmtime(eta_seconds))

        print(f"[{done}/{len(df)}] | bisherige Zeit: {elapsed_str} | noch ca.: {eta_str} | Fehler: {fehler_count}")


# -------------------------------------------------------
# CSV speichern
# -------------------------------------------------------

ergebnisse_df = pd.DataFrame(ergebnisse)

output_path = "/kaggle/working/rag_tinyllama_ergebnisse.csv"
ergebnisse_df.to_csv(
    output_path,
    index=False,       # keine automatische Zeilennummerierung
    sep=",",           # Komma als Trennzeichen wie vorgegeben
    encoding="utf-8"
)

print(f"\n Fertig!")
print(f"  {len(ergebnisse_df)} Antworten gespeichert")
print(f"  {fehler_count} Fehler aufgetreten")
print(f"  Gespeichert: {output_path}")

# Erste 5 Ergebnisse zur Kontrolle anzeigen
print("\n=== Erste 5 Ergebnisse ===")
print(ergebnisse_df.head(5).to_string())

Prompt-Spalte: 'prompt'
Beispiel: Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?

Starte RAG Pipeline für 643 Fragen...
[5/643] | bisherige Zeit: 00:00:40 | noch ca.: 01:25:20 | Fehler: 0
[10/643] | bisherige Zeit: 00:01:17 | noch ca.: 01:21:30 | Fehler: 0
[15/643] | bisherige Zeit: 00:02:07 | noch ca.: 01:29:12 | Fehler: 0
[20/643] | bisherige Zeit: 00:02:47 | noch ca.: 01:27:11 | Fehler: 0
[25/643] | bisherige Zeit: 00:03:31 | noch ca.: 01:27:05 | Fehler: 0
[30/643] | bisherige Zeit: 00:04:05 | noch ca.: 01:23:41 | Fehler: 0
[35/643] | bisherige Zeit: 00:04:51 | noch ca.: 01:24:21 | Fehler: 0
[40/643] | bisherige Zeit: 00:05:36 | noch ca.: 01:24:35 | Fehler: 0
[45/643] | bisherige Zeit: 00:06:20 | noch ca.: 01:24:20 | Fehler: 0
[50/643] | bisherige Zeit: 00:07:06 | noch ca.: 01:24:17 | Fehler: 0
[55/643] | bisherige Zeit: 00:07:48 | noch ca.: 01:23:28 | Fehler: 0
[60/643] | bisherige Zeit: 00:08:30 | noch ca.: 01:22:37 | Fehler: 0
[65/643] | bisherige Zeit: 0

This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


[145/643] | bisherige Zeit: 00:18:47 | noch ca.: 01:04:33 | Fehler: 0
[150/643] | bisherige Zeit: 00:19:37 | noch ca.: 01:04:29 | Fehler: 0
[155/643] | bisherige Zeit: 00:20:21 | noch ca.: 01:04:06 | Fehler: 0
[160/643] | bisherige Zeit: 00:21:07 | noch ca.: 01:03:45 | Fehler: 0
[165/643] | bisherige Zeit: 00:21:44 | noch ca.: 01:02:59 | Fehler: 0
[170/643] | bisherige Zeit: 00:22:28 | noch ca.: 01:02:33 | Fehler: 0
[175/643] | bisherige Zeit: 00:23:13 | noch ca.: 01:02:07 | Fehler: 0
[180/643] | bisherige Zeit: 00:23:55 | noch ca.: 01:01:32 | Fehler: 0
[185/643] | bisherige Zeit: 00:24:40 | noch ca.: 01:01:06 | Fehler: 0
[190/643] | bisherige Zeit: 00:25:29 | noch ca.: 01:00:45 | Fehler: 0
[195/643] | bisherige Zeit: 00:26:17 | noch ca.: 01:00:24 | Fehler: 0
[200/643] | bisherige Zeit: 00:26:55 | noch ca.: 00:59:37 | Fehler: 0
[205/643] | bisherige Zeit: 00:27:43 | noch ca.: 00:59:13 | Fehler: 0
[210/643] | bisherige Zeit: 00:28:26 | noch ca.: 00:58:39 | Fehler: 0
[215/643] | bisherig